# Resumen del desarrollo realizado

El objetivo del proyecto es predecir si el cierre del EUR/USD de la siguiente jornada será mayor o menor que el cierre actual.

El trabajo comenzó utilizando datos diarios de Yahoo Finance. Después de limpiar los datos, crear el objetivo y calcular las variables predictoras, entrené una regresión logística que obtuvo aproximadamente un 79,5 % de accuracy. Sin embargo, al revisar el modelo descubrí que casi todo ese rendimiento dependía de la variable `Posicion_cierre`. Al eliminarla, el accuracy cayó hasta aproximadamente un 47 %, y además una regla muy sencilla basada únicamente en esta variable conseguía casi el mismo resultado que el modelo completo.

Como este comportamiento parecía sospechoso, repetí la comprobación utilizando datos independientes de Alpha Vantage. Con esta nueva fuente, la misma relación de `Posicion_cierre` obtuvo resultados cercanos al 50 %, por lo que el resultado anterior de Yahoo Finance no se consideró robusto. A partir de este punto decidí utilizar Alpha Vantage como fuente principal para el desarrollo del modelo y conservar Yahoo únicamente como parte del análisis de calidad de los datos.

Con Alpha Vantage trabajé con 5000 jornadas diarias comprendidas entre 2007 y 2026. Los datos no presentaban fechas duplicadas ni valores faltantes en las columnas principales. Después de calcular las variables técnicas y eliminar las primeras filas que todavía no tenían suficiente historial, el dataset quedó con 4980 observaciones.

Se utilizaron 15 variables predictoras relacionadas con retornos anteriores, características de las velas, medias móviles, volatilidad, RSI y MACD. Los datos se dividieron temporalmente en entrenamiento, validación, prueba final y una última jornada reservada para realizar una predicción futura. La prueba final, correspondiente al periodo desde 2025, todavía no se ha utilizado.

Como referencia inicial evalué dos modelos muy sencillos. La clase mayoritaria obtuvo aproximadamente un 50 % de balanced accuracy y el modelo de persistencia también permaneció alrededor del 50 %, por lo que ninguno mostró una capacidad predictiva útil.

La regresión logística fue el primer modelo que mejoró estos resultados. Con todas las variables obtuvo en validación un 53,45 % de accuracy, un 53,67 % de balanced accuracy y un 54,20 % de ROC-AUC.

También repetí el modelo eliminando `Posicion_cierre`. El resultado prácticamente no cambió, lo que confirmó que el problema encontrado anteriormente con Yahoo Finance ya no estaba presente en los datos de Alpha Vantage.

Al revisar los coeficientes de la regresión, las variables con mayor influencia fueron `Cuerpo_vela` y `Retorno_diario`, mientras que `Posicion_cierre` quedó entre las variables con menor peso.

Después analicé el rendimiento por años. En 2023 la regresión funcionó mejor, con una balanced accuracy de aproximadamente 57 %, mientras que en 2024 el resultado cayó nuevamente cerca del 50 %. Esto mostró que la pequeña capacidad predictiva encontrada no se mantenía con la misma fuerza durante todos los periodos.

Posteriormente probé Random Forest. El modelo base consiguió resultados perfectos en entrenamiento, pero solamente alrededor del 47 % en validación. Esto mostró un sobreajuste muy fuerte. Después de ajustar sus parámetros utilizando divisiones temporales, el sobreajuste disminuyó, pero el resultado de validación continuó alrededor del 47 %, por lo que Random Forest no mejoró el modelo.

También probé XGBoost. El comportamiento fue parecido: la configuración inicial obtuvo casi un 100 % en entrenamiento, pero cayó por debajo del 50 % en validación. Después de ajustar el modelo, el sobreajuste se redujo, pero la balanced accuracy de validación fue aproximadamente del 45 %. Por tanto, XGBoost tampoco consiguió encontrar una relación que se mantuviera correctamente en datos posteriores.

Finalmente realicé un ajuste adicional de la regresión logística utilizando divisiones temporales dentro del entrenamiento. La mejor configuración utilizó `C = 0.01` y no necesitó aplicar pesos diferentes a las clases.

La versión que obtuvo el mejor resultado fue la regresión logística sin `Posicion_cierre`. Sus resultados en validación fueron aproximadamente:

- Accuracy: 53,45 %.
- Balanced accuracy: 53,72 %.
- ROC-AUC: 53,51 %.

En 2023 obtuvo una balanced accuracy de 56,71 %, mientras que en 2024 obtuvo 50,78 %.

Por tanto, hasta este punto la regresión logística ha sido el modelo que mejor ha generalizado de todos los probados. Random Forest y XGBoost consiguieron aprender mucho mejor los datos de entrenamiento, pero ese rendimiento no se mantuvo en periodos posteriores.

El resultado actual no permite afirmar que exista una capacidad predictiva fuerte o completamente estable. Sin embargo, sí muestra que existe una pequeña mejora frente a los modelos base y, sobre todo, que los modelos más complejos no necesariamente producen mejores resultados cuando las relaciones presentes en el mercado cambian con el tiempo.

La prueba final desde 2025 todavía permanece sin utilizar, por lo que los resultados obtenidos hasta este punto corresponden únicamente al desarrollo y validación de los modelos.